In [10]:
from pathlib import Path
import sys
path = str(Path(Path('File.py').parent.absolute()).parent.absolute())
sys.path.insert(0, path)

import numpy as np
import matplotlib as mlab
import matplotlib.pyplot as plt
import xarray as xr
import os as os
import glob

from tools.File_tools import write_forcing

from tools.FV3_tools import read_solo_fields
from tools.CM1_tools import read_cm1_fields

_retrievebeta = "/home/louis.wicker/p_decomp/run/retrievebeta.exe"
_retrievepb = "/home/louis.wicker/p_decomp/run/retrievepb.exe"

zlevels = 10.0 + 100.*np.arange(200)

# Input directories

dirs    = {
           "solo": "/work/wicker/climate_runs/FV3_Solo/bubble/euler",
           "cm1":  "/work/wicker/climate_runs/cm1r20.3/run/bubble",
          }



exp     = { "solo": "3km" }

exp     = { "cm1": "3km", "solo": "3km" }

exp     = { "cm1": "3km", "solo": "3km" }
allcape = ( "QV14", )

for key in exp:

   for shear in ["1"]:
        for cape in allcape:
        
            label = f"{cape}" # _{shear}
        
            infile = str(os.path.join(dirs[key], "%s/%s" % (exp[key], label)))
            bfile = str(os.path.join(dirs[key], "%s/%s/buoy.nc" % (exp[key], label)))
            dfile = str(os.path.join(dirs[key], "%s/%s/den.nc" % (exp[key], label)))
            pbfile = str(os.path.join(dirs[key], "%s/%s/pb.nc" % (exp[key], label)))
            befile = str(os.path.join(dirs[key], "%s/%s/beta.nc" % (exp[key], label)))

            print(f"Reading history file:        {infile}")
            print(f"Name of output density file: {dfile}")
            print(f"Name of output buoyancy file: {bfile}")
            print(f"Name of output beta    file: {befile}")
            print(f"Name of output Pb      file: {pbfile}")

            if key == 'cm1':

                ds = read_cm1_fields(infile, vars = ['buoy','den'], zinterp=zlevels )

                write_forcing(ds, dfile, variable='den')
                write_forcing(ds, bfile, variable='rbuoy')
        
            if key == 'solo':

                ds = read_solo_fields(infile, vars = ['buoy', 'den'], zinterp=zlevels )

                write_forcing(ds, dfile, variable='den')
                write_forcing(ds, bfile, variable='rbuoy')

            print("\n Wrote out beta forcing file:  %s" % dfile)
            print("\n Wrote out Pb forcing file:    %s" % bfile)

            cmd = "%s -i %s -o %s" % (_retrievebeta, dfile, befile)
            print(" \n Running cmd: %s" % cmd)
            ret = os.system(cmd)

            cmd = "%s -i %s -o %s" % (_retrievepb, bfile, pbfile)
            print(" \n Running cmd: %s" % cmd)
            ret = os.system(cmd)

            if ret == 0:
                print("\n Beta solution file  %s succesfully write" % befile)
            else:
                print("\n Beta solution did not execute....")
                sys.exit(1)
# the end


Reading history file:        /work/wicker/climate_runs/cm1r20.3/run/bubble/3km/QV14
Name of output density file: /work/wicker/climate_runs/cm1r20.3/run/bubble/3km/QV14/den.nc
Name of output buoyancy file: /work/wicker/climate_runs/cm1r20.3/run/bubble/3km/QV14/buoy.nc
Name of output beta    file: /work/wicker/climate_runs/cm1r20.3/run/bubble/3km/QV14/beta.nc
Name of output Pb      file: /work/wicker/climate_runs/cm1r20.3/run/bubble/3km/QV14/pb.nc
------------------------------------------------------------------------------------------------------------------------

 Now reading... /work/wicker/climate_runs/cm1r20.3/run/bubble/3km/QV14/cm1out.nc

 Time for xarray to load:  /work/wicker/climate_runs/cm1r20.3/run/bubble/3km/QV14/cm1out.nc: 0.24 sec 

 Completed setup for:  /work/wicker/climate_runs/cm1r20.3/run/bubble/3km/QV14/cm1out.nc --> time to initialize variables: 0.68 sec 

 Completed reading in:  /work/wicker/climate_runs/cm1r20.3/run/bubble/3km/QV14/cm1out.nc --> time to process 